In [1]:
import numpy as np
import pandas as pd
import statsmodels.api as sm
import plotly.graph_objects as go
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import KFold
from sklearn.metrics import mean_squared_error, r2_score

In [2]:
df = pd.read_csv("img_attribute.csv")
df = df.dropna(subset=['Absolute_Humidity(g/m^3)', 'Temperature(C)'])

label_dummies = pd.get_dummies(df['label'], prefix='Cluster')
df = pd.concat([df.drop(columns=['label']), label_dummies], axis=1)
label_cols = [c for c in df.columns if c.startswith('Cluster_')]

features = ['Cirrus_weighted', 'Cirrostratus_weighted', 'Stratus_weighted',
            'Stratocumulus_weighted', 'Cumulus_weighted', 'Cirrocumulus_weighted',
            'Nimbus_weighted', 'Clear_weighted', 'Absolute_Humidity(g/m^3)']

# features = ['Cirrus', 'Cirrostratus', 'Stratus',
#             'Stratocumulus', 'Cumulus', 'Cirrocumulus',
#             'Nimbus', 'Clear', 'Absolute_Humidity(g/m^3)']

target = 'Radiation Value'
X, y = df[features], df[target]
X.columns = X.columns.str.replace('_weighted', '')
X.columns = X.columns.str.replace('Absolute_Humidity(g/m^3)', 'AbsHumidity', regex=False)
X, y = X.astype(float), y.astype(float)

In [3]:
model = LinearRegression()

kf = KFold(n_splits=5, shuffle=True, random_state=42)

r2_scores = []
rmse_scores = []

for train_idx, test_idx in kf.split(X):
    X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
    y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]

    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)

    r2_scores.append(r2_score(y_test, y_pred))
    rmse_scores.append(np.sqrt(mean_squared_error(y_test, y_pred)))

print(f"Average R²:   {np.mean(r2_scores):.4f}")
print(f"Average RMSE: {np.mean(rmse_scores):.4f}")

Average R²:   0.7659
Average RMSE: 4.6280


In [4]:
X_const = sm.add_constant(X)
model_sm = sm.OLS(y, X_const).fit()
print(model_sm.summary())

                            OLS Regression Results                            
Dep. Variable:        Radiation Value   R-squared:                       0.767
Model:                            OLS   Adj. R-squared:                  0.767
Method:                 Least Squares   F-statistic:                     2279.
Date:                Wed, 06 May 2026   Prob (F-statistic):               0.00
Time:                        12:20:39   Log-Likelihood:                -18360.
No. Observations:                6226   AIC:                         3.674e+04
Df Residuals:                    6216   BIC:                         3.681e+04
Df Model:                           9                                         
Covariance Type:            nonrobust                                         
                    coef    std err          t      P>|t|      [0.025      0.975]
---------------------------------------------------------------------------------
const           -31.0424      0.531    -58.424

In [5]:
# df_test = pd.read_csv("img_attribute.csv")
df_test = pd.read_csv("img_attribute_case.csv")
df_test = df_test.dropna(subset=['Absolute_Humidity(g/m^3)'])

label_dummies = pd.get_dummies(df_test['label'], prefix='Cluster')
df_test = pd.concat([df_test.drop(columns=['label']), label_dummies], axis=1)
label_cols = [c for c in df_test.columns if c.startswith('Cluster_')]

features = ['Cirrus_weighted', 'Cirrostratus_weighted', 'Stratus_weighted',
            'Stratocumulus_weighted', 'Cumulus_weighted', 'Cirrocumulus_weighted',
            'Nimbus_weighted', 'Clear_weighted', 'Absolute_Humidity(g/m^3)']

# features = ['Cirrus', 'Cirrostratus', 'Stratus',
#             'Stratocumulus', 'Cumulus', 'Cirrocumulus',
#             'Nimbus', 'Clear', 'Absolute_Humidity(g/m^3)']

target = 'Radiation Value'
X_test, y_test = df_test[features], df_test[target]
X_test.columns = X_test.columns.str.replace('_weighted', '')
X_test.columns = X_test.columns.str.replace('Absolute_Humidity(g/m^3)', 'AbsHumidity', regex=False)
X_test, y_test = X_test.astype(float), y_test.astype(float)

In [6]:
X_test_const = sm.add_constant(X_test)
y_pred_sm = model_sm.predict(X_test_const)
rmse_sm = np.sqrt(mean_squared_error(y_test, y_pred_sm))
r2_score_sm = r2_score(y_test, y_pred_sm)
print(f"Statsmodels R²:   {r2_score_sm:.4f}")
print(f"Statsmodels RMSE: {rmse_sm:.4f}")

Statsmodels R²:   0.7210
Statsmodels RMSE: 5.6942


### Performance plot

In [7]:
x, y = y_pred_sm, y_test

fig = go.Figure()
fig.add_trace(
    go.Scatter(
        x=x, y=y,
        mode='markers',
        marker=dict(size=15, opacity=0.6, color='#2a9d8f'),
        # marker=dict(size=15, opacity=0.6, color='#39BCBC'),
        name='Data',
        showlegend=False
    )
)

fig.add_trace(
    go.Scatter(
        x=[-40, 5], y=[-40, 5],
        mode='lines',
        line=dict(color='Gray', width=10),
        showlegend=False
    )
)

fig.add_annotation(
    x=3, y=1,
    text="x = y",
    showarrow=False,
    font=dict(size=56, weight='bold', family="Arial", color="gray"),
    textangle=-45
)

fig.add_annotation(
    x=0.03, y=0.99,
    xref="paper", yref="paper",
    text="a)",
    showarrow=False,
    font=dict(size=70, weight='bold', family="Arial", color="black"),
    xanchor="left",
    yanchor="top"
)

fig.update_layout(
    height=1200,
    width=1400,
    xaxis = dict(
        title = dict(text='Predicted', standoff=30),
        tickfont=dict(size=56, family='Arial', weight='bold'),
        titlefont=dict(size=64, family='Arial', weight='bold'),
        linewidth=6, linecolor='black',
        range=[-40, 5],
        tickwidth=6, ticklen=15
    ),
    yaxis = dict(
        title = dict(text='Ground Truth', standoff=30),
        tickfont=dict(size=56, family='Arial', weight='bold'),
        titlefont=dict(size=64, family='Arial', weight='bold'),
        linewidth=6, linecolor='black',
        range=[-40, 5],
        tickwidth=6, ticklen=15
    ),
    margin=dict(t=10, b=180, l=220, r=10),
    template='simple_white'
)

fig.show()

In [8]:
df_test['Predicted'] = y_pred_sm
df_test['Residual'] = (df_test[target] - df_test['Predicted'])

In [9]:
# calculate daily averages
required_cols = {'Year','Month','Day', target, 'Predicted', 'Residual'}
daily = (df_test.groupby(['Year','Month','Day'], as_index=False)
         .agg({target:'mean', 'Predicted':'mean', 'Residual':'mean'}))
daily = daily.rename(columns={target: 'Actual'})

daily['Date'] = pd.to_datetime(daily[['Year','Month','Day']])
daily = daily.sort_values('Date').reset_index(drop=True)
full_dates = pd.date_range(daily['Date'].min(), daily['Date'].max(), freq='D')
daily = (daily.set_index('Date').reindex(full_dates).rename_axis('Date').reset_index())

In [10]:
fig = go.Figure()
fig.add_trace(go.Scatter(x=daily['Date'], y=daily['Residual'], mode='lines+markers',
                         name='Residual', line=dict(color='#5f9ea0', width=10), 
                         marker=dict(size=25), connectgaps=False)) #39BCBC

fig.add_shape(type="line", x0=daily['Date'].min(), x1=daily['Date'].max(), y0=5, y1=5,
              line=dict(color="gray", width=4, dash="dash"))
fig.add_shape(type="line", x0=daily['Date'].min(), x1=daily['Date'].max(), y0=-5, y1=-5,
              line=dict(color="gray", width=4, dash="dash"))

if daily['Date'].notna().any():
    x_min = daily['Date'].min()
    x_max = daily['Date'].max()
    span = x_max - x_min
    pad = span * 0.02  # 2% padding
    fig.update_xaxes(range=[x_min - pad, x_max + pad])
    fig.update_yaxes(range=[daily['Residual'].min() - 3, daily['Residual'].max() + 3])

fig.update_xaxes(nticks=10)
fig.update_yaxes(nticks=5)

fig.update_layout(
    xaxis=dict(title=dict(text='Date', font=dict(size=64, family='Arial', weight='bold'), standoff=40), tickformat="%m/%d", tickwidth=6, ticklen=15),
    yaxis=dict(title=dict(text='Residual', font=dict(size=64, family='Arial', weight='bold'), standoff=40), tickwidth=6, ticklen=15),
    template='simple_white',
    margin=dict(l=70, r=20, t=10, b=50),
    height=800, width=2000
)

fig.update_xaxes(tickfont=dict(size=64, family='Arial', weight='bold'), linewidth=6, linecolor='black')
fig.update_yaxes(tickfont=dict(size=64, family='Arial', weight='bold'), linewidth=6, linecolor='black')

fig.show()